In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "d320018a",
   "metadata": {},
   "source": [
    "# 02 · Silver (local + pandas) — limpo, tipado, deduplicado\n",
    "\n",
    "```\n",
    "BRONZE (Parquet no bucket warehouse) ──► SILVER (Parquet no bucket warehouse) + QUARENTENA\n",
    "```\n",
    "\n",
    "Tudo o que a Silver faz tem uma **regra explícita**. O que viola a regra não some: vai para a **quarentena**.\n",
    "\n",
    "| Tabela | Regras |\n",
    "|---|---|\n",
    "| clientes | `cliente_id` int; `estado` em maiúsculas; e-mail em minúsculas, inválido vira `NULL`; datas em vários formatos; fica a versão mais recente por `cliente_id` |\n",
    "| produtos | `preco` e `custo` > 0, senão quarentena |\n",
    "| pedidos | tipos corretos; deduplica por `pedido_id`; **cliente inexistente → quarentena**; `itens` explodem em `itens_pedido` |\n",
    "| avaliacoes | `nota` entre 1 e 5, senão quarentena; deduplica por `avaliacao_id` |\n",
    "\n",
    "A Silver é reconstruída a partir de **toda** a Bronze (todos os lotes), então é segura para reexecutar.\n",
    "\n",
    "**Diferença para a versão Spark:** aqui são arquivos Parquet no bucket, **não tabelas Iceberg**. Não existe `snapshots` nem `VERSION AS OF`. No final usaremos o **versionamento do bucket**, que é o recurso do próprio object storage para guardar o histórico.\n",
    "\n",
    "**Antes de começar:** rode o `01_bronze_local_pandas.ipynb` (a Bronze precisa existir) com o MinIO no ar."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "15486116",
   "metadata": {},
   "source": [
    "## Passo 0 · Conexão e funções de apoio"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9ea33584",
   "metadata": {},
   "outputs": [],
   "source": [
    "import io\n",
    "import json\n",
    "import os\n",
    "from datetime import datetime, timezone\n",
    "\n",
    "import boto3\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "\n",
    "ENDPOINT = os.environ.get(\"MINIO_ENDPOINT\", \"http://localhost:9000\")\n",
    "USUARIO = os.environ.get(\"MINIO_USER\", \"admin\")\n",
    "SENHA = os.environ.get(\"MINIO_PASSWORD\", \"admin12345\")\n",
    "\n",
    "s3 = boto3.client(\"s3\", endpoint_url=ENDPOINT, aws_access_key_id=USUARIO,\n",
    "                  aws_secret_access_key=SENHA, region_name=\"us-east-1\")\n",
    "\n",
    "\n",
    "def listar(bucket, prefixo=\"\"):\n",
    "    achados = []\n",
    "    for pagina in s3.get_paginator(\"list_objects_v2\").paginate(Bucket=bucket, Prefix=prefixo):\n",
    "        achados += [(o[\"Key\"], o[\"Size\"]) for o in pagina.get(\"Contents\", [])]\n",
    "    return achados\n",
    "\n",
    "\n",
    "def ler_prefixo(bucket, prefixo):\n",
    "    \"\"\"Lê todos os Parquet de um prefixo e junta num DataFrame.\"\"\"\n",
    "    partes = []\n",
    "    for chave, _ in listar(bucket, prefixo):\n",
    "        corpo = s3.get_object(Bucket=bucket, Key=chave)[\"Body\"].read()\n",
    "        partes.append(pd.read_parquet(io.BytesIO(corpo)))\n",
    "    return pd.concat(partes, ignore_index=True)\n",
    "\n",
    "\n",
    "def apagar_prefixo(bucket, prefixo):\n",
    "    for chave, _ in listar(bucket, prefixo):\n",
    "        s3.delete_object(Bucket=bucket, Key=chave)\n",
    "\n",
    "\n",
    "def gravar_silver(df, tabela):\n",
    "    prefixo = f\"silver/{tabela}/\"\n",
    "    apagar_prefixo(\"warehouse\", prefixo)            # reconstruída por inteiro: reexecutar não duplica\n",
    "    buffer = io.BytesIO()\n",
    "    df.to_parquet(buffer, index=False, engine=\"pyarrow\")\n",
    "    s3.put_object(Bucket=\"warehouse\", Key=f\"{prefixo}part-0000.parquet\", Body=buffer.getvalue())\n",
    "    return len(df)\n",
    "\n",
    "\n",
    "def para_int(serie):\n",
    "    return pd.to_numeric(serie, errors=\"coerce\").astype(\"Int64\")\n",
    "\n",
    "\n",
    "print(\"conectado em\", ENDPOINT)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7084c9b8",
   "metadata": {},
   "source": [
    "## Passo 1 · Ler a Bronze\n",
    "A Bronze guarda **tudo como veio**: CSVs em texto e pedidos duplicados. A Silver parte daqui."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4dfccc60",
   "metadata": {},
   "outputs": [],
   "source": [
    "brz = {t: ler_prefixo(\"warehouse\", f\"bronze/{t}/\") for t in (\"clientes\", \"produtos\", \"pedidos\", \"avaliacoes\")}\n",
    "for t, df in brz.items():\n",
    "    print(f\"{t:<11} {len(df):>7,} linhas | colunas: {list(df.columns)[:8]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "61d99cab",
   "metadata": {},
   "source": [
    "## Antes da função: faça na mão\n",
    "Na aula de Spark você explodiu a coluna `itens`. Agora vamos fazer isso **a partir da Bronze**, que tem uma diferença: ela guarda os pedidos duplicados.\n",
    "Se explodirmos antes de deduplicar, cada item duplicado conta duas vezes e o faturamento fica inflado."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "041676f2",
   "metadata": {},
   "outputs": [],
   "source": [
    "def deduplicar(df, chave):\n",
    "    \"\"\"Fica com a versão mais recente (_ingest_ts) de cada chave.\"\"\"\n",
    "    return df.sort_values(\"_ingest_ts\").drop_duplicates(chave, keep=\"last\")\n",
    "\n",
    "\n",
    "def explodir_itens(pedidos):\n",
    "    \"\"\"Uma linha por item de pedido. item_seq guarda a posição do item dentro do pedido.\"\"\"\n",
    "    base = pedidos[[\"pedido_id\", \"cliente_id\", \"data_pedido\", \"status\", \"itens\"]].copy()\n",
    "    base[\"itens\"] = base[\"itens\"].apply(lambda x: list(x) if isinstance(x, (list, np.ndarray)) else [])\n",
    "    base = base.explode(\"itens\").dropna(subset=[\"itens\"]).reset_index(drop=True)   # explode repete o índice\n",
    "    base[\"item_seq\"] = base.groupby(\"pedido_id\").cumcount()\n",
    "    det = pd.DataFrame(base[\"itens\"].tolist()).reindex(columns=[\"produto_id\", \"qtd\", \"preco_unit\"])\n",
    "    itens = base.drop(columns=\"itens\").join(det)\n",
    "    itens[\"produto_id\"] = para_int(itens[\"produto_id\"])\n",
    "    itens[\"qtd\"] = pd.to_numeric(itens[\"qtd\"], errors=\"coerce\")\n",
    "    itens[\"preco_unit\"] = pd.to_numeric(itens[\"preco_unit\"], errors=\"coerce\")\n",
    "    itens[\"valor_item\"] = (itens[\"qtd\"] * itens[\"preco_unit\"]).round(2)\n",
    "    return itens\n",
    "\n",
    "\n",
    "print(\"Bronze:\", len(brz[\"pedidos\"]), \"linhas |\", brz[\"pedidos\"][\"pedido_id\"].nunique(), \"pedidos distintos\")\n",
    "\n",
    "pedidos_unicos = deduplicar(brz[\"pedidos\"], \"pedido_id\")\n",
    "print(\"Duplicados descartados:\", len(brz[\"pedidos\"]) - len(pedidos_unicos))\n",
    "\n",
    "itens = explodir_itens(pedidos_unicos)\n",
    "print(\"pedidos:\", len(pedidos_unicos), \"| itens:\", len(itens))\n",
    "itens.head(5)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "37745fa1",
   "metadata": {},
   "source": [
    "### Checagens de qualidade nos itens\n",
    "Mesmas três perguntas da aula (valores inválidos, chave duplicada, produto inexistente), mas agora quem falha **não some**: ganha um motivo e vai para a quarentena."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "dca701e4",
   "metadata": {},
   "outputs": [],
   "source": [
    "ids_produtos = set(para_int(brz[\"produtos\"][\"produto_id\"]).dropna())\n",
    "\n",
    "\n",
    "def separar_itens(itens, ids_produtos):\n",
    "    valor_ok = (itens[\"qtd\"] > 0) & (itens[\"preco_unit\"] > 0) & itens[\"produto_id\"].notna()\n",
    "    existe = itens[\"produto_id\"].isin(ids_produtos)\n",
    "    motivo = np.select([~valor_ok, ~existe], [\"valor_invalido\", \"produto_inexistente\"], default=\"\")\n",
    "    return itens[motivo == \"\"].copy(), itens[motivo != \"\"].assign(motivo=motivo[motivo != \"\"])\n",
    "\n",
    "\n",
    "itens_ok, itens_rej = separar_itens(itens, ids_produtos)\n",
    "\n",
    "print(\"itens válidos:\", len(itens_ok), \"| rejeitados:\", len(itens_rej))\n",
    "print(itens_rej[\"motivo\"].value_counts().to_string() if len(itens_rej) else \"nenhum rejeitado\")\n",
    "\n",
    "# chave composta única (pedido_id, item_seq) nos válidos\n",
    "dup = int(itens_ok.duplicated([\"pedido_id\", \"item_seq\"]).sum())\n",
    "print(\"chaves duplicadas:\", dup, \"->\", \"OK\" if dup == 0 else \"ERRO\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "78ad87cd",
   "metadata": {},
   "source": [
    "### Salvar na Silver"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "b523aeff",
   "metadata": {},
   "outputs": [],
   "source": [
    "n = gravar_silver(itens_ok, \"itens_pedido_aula\")\n",
    "print(n, \"linhas em s3://warehouse/silver/itens_pedido_aula/\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "64e29c66",
   "metadata": {},
   "source": [
    "## Passo 2 · O resto da Silver\n",
    "Agora as outras tabelas, com as mesmas ideias: tipar, padronizar, deduplicar e mandar para a quarentena o que viola a regra.\n",
    "A função reaproveita `deduplicar`, `explodir_itens` e `separar_itens` que você acabou de escrever."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "7b450ccf",
   "metadata": {},
   "outputs": [],
   "source": [
    "EMAIL_RE = r\"^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$\"\n",
    "TS = datetime.now(timezone.utc)\n",
    "\n",
    "\n",
    "def _json(o):\n",
    "    return o.tolist() if hasattr(o, \"tolist\") else str(o)\n",
    "\n",
    "\n",
    "def para_quarentena(df, origem):\n",
    "    colunas = [\"origem\", \"registro\", \"_quarentena_ts\"]\n",
    "    if df.empty:\n",
    "        return pd.DataFrame(columns=colunas)\n",
    "    registros = [json.dumps(r, default=_json, ensure_ascii=False) for r in df.to_dict(\"records\")]\n",
    "    return pd.DataFrame({\"origem\": origem, \"registro\": registros, \"_quarentena_ts\": TS})\n",
    "\n",
    "\n",
    "def build_silver():\n",
    "    quarentena = []\n",
    "\n",
    "    # clientes\n",
    "    cli = brz[\"clientes\"].copy()\n",
    "    cli[\"cliente_id\"] = para_int(cli[\"cliente_id\"])\n",
    "    cli[\"estado\"] = cli[\"estado\"].str.strip().str.upper()\n",
    "    cli[\"email\"] = cli[\"email\"].str.strip().str.lower()\n",
    "    cli.loc[~cli[\"email\"].fillna(\"\").str.match(EMAIL_RE), \"email\"] = None      # inválido vira NULL\n",
    "    for c in [c for c in cli.columns if c.startswith(\"data\")]:\n",
    "        cli[c] = pd.to_datetime(cli[c], format=\"mixed\", dayfirst=True, errors=\"coerce\")\n",
    "    cli = deduplicar(cli, \"cliente_id\")\n",
    "\n",
    "    # produtos\n",
    "    prod = brz[\"produtos\"].copy()\n",
    "    prod[\"produto_id\"] = para_int(prod[\"produto_id\"])\n",
    "    for c in (\"preco\", \"custo\"):\n",
    "        prod[c] = pd.to_numeric(prod[c], errors=\"coerce\")\n",
    "    invalido = ~((prod[\"preco\"] > 0) & (prod[\"custo\"] > 0))\n",
    "    quarentena.append(para_quarentena(prod[invalido], \"produtos\"))\n",
    "    prod = deduplicar(prod[~invalido], \"produto_id\")\n",
    "\n",
    "    # pedidos\n",
    "    ped = deduplicar(brz[\"pedidos\"], \"pedido_id\").copy()\n",
    "    ped[\"cliente_id\"] = para_int(ped[\"cliente_id\"])\n",
    "    ped[\"data_pedido\"] = pd.to_datetime(ped[\"data_pedido\"], format=\"mixed\", errors=\"coerce\")\n",
    "    orfao = ~ped[\"cliente_id\"].isin(set(cli[\"cliente_id\"].dropna()))\n",
    "    quarentena.append(para_quarentena(ped[orfao], \"pedidos_cliente_orfao\"))\n",
    "    ped = ped[~orfao]\n",
    "\n",
    "    # itens dos pedidos válidos\n",
    "    ok, rej = separar_itens(explodir_itens(ped), set(prod[\"produto_id\"].dropna()))\n",
    "    for motivo, grupo in rej.groupby(\"motivo\"):\n",
    "        quarentena.append(para_quarentena(grupo.drop(columns=\"motivo\"), f\"itens_{motivo}\"))\n",
    "    ped = ped.drop(columns=\"itens\")\n",
    "\n",
    "    # avaliacoes\n",
    "    av = brz[\"avaliacoes\"].copy()\n",
    "    av[\"nota\"] = pd.to_numeric(av[\"nota\"], errors=\"coerce\")\n",
    "    for c in (\"cliente_id\", \"produto_id\"):\n",
    "        if c in av.columns:\n",
    "            av[c] = para_int(av[c])\n",
    "    ruim = ~av[\"nota\"].between(1, 5)\n",
    "    quarentena.append(para_quarentena(av[ruim], \"avaliacoes_nota_invalida\"))\n",
    "    av = deduplicar(av[~ruim], \"avaliacao_id\")\n",
    "\n",
    "    tabelas = {\"clientes\": cli, \"produtos\": prod, \"pedidos\": ped, \"itens_pedido\": ok, \"avaliacoes\": av,\n",
    "               \"quarentena\": pd.concat(quarentena, ignore_index=True)}\n",
    "    return {nome: gravar_silver(df, nome) for nome, df in tabelas.items()}\n",
    "\n",
    "\n",
    "resumo = build_silver()\n",
    "resumo"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b4681cc7",
   "metadata": {},
   "source": [
    "## Reconciliação: tudo que entrou tem que estar em algum lugar\n",
    "Regra de ouro: **entrada = saída + quarentena**. Se a conta não fecha, o pipeline está perdendo dado sem avisar."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "21abc83d",
   "metadata": {},
   "outputs": [],
   "source": [
    "sil_ped = ler_prefixo(\"warehouse\", \"silver/pedidos/\")\n",
    "quar = ler_prefixo(\"warehouse\", \"silver/quarentena/\")\n",
    "\n",
    "n_bronze = brz[\"pedidos\"][\"pedido_id\"].nunique()\n",
    "n_silver = len(sil_ped)\n",
    "n_quar = int((quar[\"origem\"] == \"pedidos_cliente_orfao\").sum())\n",
    "print(f\"pedidos : {n_bronze} = {n_silver} (silver) + {n_quar} (quarentena) -> {n_bronze == n_silver + n_quar}\")\n",
    "\n",
    "n_mao = len(ler_prefixo(\"warehouse\", \"silver/itens_pedido_aula/\"))\n",
    "n_funcao = len(ler_prefixo(\"warehouse\", \"silver/itens_pedido/\"))\n",
    "print(f\"itens   : feito na mão = {n_mao} | feito pela função = {n_funcao}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c49c250f",
   "metadata": {},
   "source": [
    "**Pergunta:** se os dois números de itens forem diferentes, por quê? Na versão à mão explodimos todos os pedidos; a função só explode os pedidos cujo cliente existe. Os itens dos pedidos órfãos ficaram de fora."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fb7b715b",
   "metadata": {},
   "source": [
    "## O que foi para a quarentena?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9a8f1641",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(quar[\"origem\"].value_counts().to_string())\n",
    "quar.head(5)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b441e97a",
   "metadata": {},
   "source": [
    "## Data Quality: medir, não só limpar\n",
    "Cada regra devolve uma linha no relatório. Guardar o relatório no storage é o começo da **observabilidade**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "44fb95cd",
   "metadata": {},
   "outputs": [],
   "source": [
    "resultados = []\n",
    "EXEC = datetime.now(timezone.utc)\n",
    "\n",
    "\n",
    "def regra(df, dataset, nome, coluna, falhou, severidade=\"erro\"):\n",
    "    n = int(falhou.sum())\n",
    "    status = \"OK\" if n == 0 else (\"AVISO\" if severidade == \"aviso\" else \"ERRO\")\n",
    "    resultados.append({\"dataset\": dataset, \"regra\": nome, \"coluna\": coluna, \"total\": len(df), \"falhas\": n,\n",
    "                       \"pct_falha\": round(100 * n / max(len(df), 1), 2), \"status\": status, \"executado_em\": EXEC})\n",
    "\n",
    "\n",
    "cli = ler_prefixo(\"warehouse\", \"silver/clientes/\")\n",
    "ped = ler_prefixo(\"warehouse\", \"silver/pedidos/\")\n",
    "av = ler_prefixo(\"warehouse\", \"silver/avaliacoes/\")\n",
    "\n",
    "regra(cli, \"silver.clientes\", \"not_null\", \"cliente_id\", cli[\"cliente_id\"].isna())\n",
    "regra(cli, \"silver.clientes\", \"unique\", \"cliente_id\", cli[\"cliente_id\"].duplicated())\n",
    "regra(cli, \"silver.clientes\", \"not_null\", \"email\", cli[\"email\"].isna(), \"aviso\")\n",
    "regra(ped, \"silver.pedidos\", \"not_null\", \"pedido_id\", ped[\"pedido_id\"].isna())\n",
    "regra(ped, \"silver.pedidos\", \"unique\", \"pedido_id\", ped[\"pedido_id\"].duplicated())\n",
    "regra(ped, \"silver.pedidos\", \"in_set\", \"status\",\n",
    "      ~ped[\"status\"].isin([\"pago\", \"enviado\", \"entregue\", \"cancelado\", \"devolvido\"]))\n",
    "regra(av, \"silver.avaliacoes\", \"unique\", \"avaliacao_id\", av[\"avaliacao_id\"].duplicated())\n",
    "regra(av, \"silver.avaliacoes\", \"between\", \"nota\", ~av[\"nota\"].between(1, 5))\n",
    "\n",
    "relatorio = pd.DataFrame(resultados)\n",
    "\n",
    "# cada execução grava um objeto NOVO (nunca editamos um objeto existente): o histórico fica no prefixo\n",
    "chave = f\"silver/dq_resultados/{EXEC:%Y%m%dT%H%M%S}.parquet\"\n",
    "buf = io.BytesIO()\n",
    "relatorio.to_parquet(buf, index=False)\n",
    "s3.put_object(Bucket=\"warehouse\", Key=chave, Body=buf.getvalue())\n",
    "print(\"relatório gravado em\", chave, \"| execuções guardadas:\", len(listar(\"warehouse\", \"silver/dq_resultados/\")))\n",
    "relatorio"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c78b5422",
   "metadata": {},
   "source": [
    "Se alguma regra aparece com status `AVISO` ou `ERRO`, ótimo: é o ponto de conversa. (Dica: e-mail inválido vira `NULL` na Silver, e a regra `not_null(email)` aparece como `AVISO`: o cliente continua existindo, só perdeu o contato.)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f7517126",
   "metadata": {},
   "source": [
    "## Histórico de versões: a Silver tem memória?\n",
    "Com Iceberg, cada escrita cria um *snapshot*. Aqui não há Iceberg, mas o próprio **object storage** pode guardar versões de cada objeto, quando o **versionamento do bucket** está ligado.\n",
    "Vamos criar um bucket só para este experimento, ligar o versionamento e repetir o erro humano: alguém \"limpa\" os pedidos cancelados achando que eram lixo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "58cffa1e",
   "metadata": {},
   "outputs": [],
   "source": [
    "BUCKET_H = \"historico\"\n",
    "CHAVE_H = \"itens_pedido/part-0000.parquet\"\n",
    "\n",
    "if BUCKET_H not in {b[\"Name\"] for b in s3.list_buckets()[\"Buckets\"]}:\n",
    "    s3.create_bucket(Bucket=BUCKET_H)\n",
    "s3.put_bucket_versioning(Bucket=BUCKET_H, VersioningConfiguration={\"Status\": \"Enabled\"})\n",
    "\n",
    "\n",
    "def gravar_versao(df):\n",
    "    buf = io.BytesIO()\n",
    "    df.to_parquet(buf, index=False)\n",
    "    return s3.put_object(Bucket=BUCKET_H, Key=CHAVE_H, Body=buf.getvalue())[\"VersionId\"]\n",
    "\n",
    "\n",
    "def ler_versao(version_id=None):\n",
    "    extra = {\"VersionId\": version_id} if version_id else {}\n",
    "    corpo = s3.get_object(Bucket=BUCKET_H, Key=CHAVE_H, **extra)[\"Body\"].read()\n",
    "    return pd.read_parquet(io.BytesIO(corpo))\n",
    "\n",
    "\n",
    "def historico():\n",
    "    r = s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)\n",
    "    return pd.DataFrame([{\"version_id\": v[\"VersionId\"], \"gravado_em\": v[\"LastModified\"],\n",
    "                          \"KB\": round(v[\"Size\"] / 1024, 1), \"atual\": v[\"IsLatest\"]}\n",
    "                         for v in r.get(\"Versions\", [])])      # mais recente primeiro\n",
    "\n",
    "\n",
    "versao_boa = gravar_versao(itens_ok)\n",
    "print(\"linhas na versão boa:\", len(itens_ok))\n",
    "historico()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "78386110",
   "metadata": {},
   "source": [
    "Agora o estrago: gravamos de novo o **mesmo objeto**, sem os cancelados."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "2a470218",
   "metadata": {},
   "outputs": [],
   "source": [
    "sem_cancelados = itens_ok[itens_ok[\"status\"] != \"cancelado\"]\n",
    "gravar_versao(sem_cancelados)\n",
    "\n",
    "print(f\"linhas: {len(itens_ok)} -> {len(sem_cancelados)}  (sumiram {len(itens_ok) - len(sem_cancelados)})\")\n",
    "print(\"versão boa:\", versao_boa)\n",
    "historico()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "0eec9e31",
   "metadata": {},
   "source": [
    "### Voltar no tempo (só ler)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a4e9fe6e",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"linhas na versão antiga :\", len(ler_versao(versao_boa)))\n",
    "print(\"linhas na versão atual  :\", len(ler_versao()))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "215bd056",
   "metadata": {},
   "source": [
    "### Desfazer (rollback)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "ce59c530",
   "metadata": {},
   "outputs": [],
   "source": [
    "# o rollback é COPIAR a versão boa por cima: isso cria uma versão nova, o histórico só cresce\n",
    "s3.copy_object(Bucket=BUCKET_H, Key=CHAVE_H,\n",
    "               CopySource={\"Bucket\": BUCKET_H, \"Key\": CHAVE_H, \"VersionId\": versao_boa})\n",
    "print(\"linhas depois do rollback:\", len(ler_versao()))\n",
    "historico()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8c5306e5",
   "metadata": {},
   "source": [
    "### O que aprendemos\n",
    "- A versão errada **não destruiu** a boa: o object storage manteve as duas. Compare com o Passo 5 da Bronze, onde o objeto era sobrescrito e o passado se perdia: a diferença é o versionamento do bucket.\n",
    "- O rollback aqui não \"volta\" nada: ele **cria uma nova versão** igual à antiga. Na tabela Iceberg o rollback muda o ponteiro da tabela. Nos dois casos o histórico é preservado.\n",
    "- Ler uma versão antiga é seguro. Reverter muda o dado para todos, então deve ser feito por quem tem responsabilidade sobre ele.\n",
    "- **Mas atenção (LGPD):** `delete_object` com versionamento só coloca uma *marca de exclusão*; as versões antigas continuam lá. Para apagar de verdade é preciso apagar cada `VersionId`. **Pergunta:** quanto tempo guardar o histórico? Ele ajuda a auditar, mas atrasa o direito ao esquecimento.\n",
    "- **Comparação com a versão Spark:** Iceberg versiona a *tabela* (várias linhas e arquivos, de forma consistente). O versionamento do bucket versiona cada *objeto*, um a um, sem saber o que é uma tabela."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "912f3783",
   "metadata": {},
   "source": [
    "## Checkpoint\n",
    "- [ ] Silver com 5 tabelas (`clientes`, `produtos`, `pedidos`, `itens_pedido`, `avaliacoes`) + `quarentena` + `dq_resultados`.\n",
    "- [ ] Você consegue explicar por que um pedido foi para a quarentena.\n",
    "- [ ] Você consegue desfazer uma mudança errada na Silver e explica por que o dado antigo ainda existia.\n",
    "\n",
    "➡️ Próximo: `03_gold_local_pandas.ipynb`"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}